# Sfumato — GUIDE triage model (true positive / false positive / benign positive)

Trains the reference model on **Microsoft GUIDE** (real SOC incidents graded by analysts) and publishes it,
signed, to your **Google Cloud Storage bucket**. Sfumato downloads it and uses it as a second opinion.

**Before running**
1. *Add Input* → search **"Microsoft Security Incident Prediction"** (dataset by Microsoft) and add it.
2. *Add-ons → Secrets*:
   - `ARTIFACT_SIGNING_KEY`: a long random string, **the same value as in Sfumato's `.env`**;
   - `GCP_SA_JSON`: the JSON key of a service account that can *write* to the bucket (`roles/storage.objectAdmin` on the bucket only).
3. Accelerator: **None (CPU)**. The model is gradient-boosted trees, a GPU brings nothing.
4. Internet: **on** (needed to reach Google Cloud Storage).

In [ ]:
!pip install -q google-cloud-storage
import os, json
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
os.environ["ARTIFACT_SIGNING_KEY"] = sec.get_secret("ARTIFACT_SIGNING_KEY")
with open("/kaggle/working/sa.json", "w") as f:
    f.write(sec.get_secret("GCP_SA_JSON"))
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/kaggle/working/sa.json"

BUCKET_URI = "gs://YOUR-BUCKET/sfumato"   # <- bucket (+ optional prefix) that receives the model
MAX_ROWS = None                           # e.g. 2_000_000 for a quick first run (full data: ~9.5M rows)

`artifacts.py` — generated from `backend/app/storage/artifacts.py` in the Sfumato repository.

In [ ]:
%%writefile artifacts.py
"""Versioned artifact stores: local folder, Google Cloud Storage (gs://) and Hugging Face Hub (hf://).

Standalone on purpose (standard library only; google-cloud-storage / huggingface_hub are imported lazily):
the very same file is embedded in the Kaggle notebooks that publish models and knowledge bases, and used
by the Sfumato backend to download them.

Layout inside a store::

    <name>/LATEST.json                    {"version": "20261112T101500Z"}
    <name>/<version>/ARTIFACT.json        {"meta": {name, version, created_at, files: {file: sha256}}, "signature": hmac}
    <name>/<version>/<files...>

Integrity: every file is checked against the sha256 in ARTIFACT.json, and ARTIFACT.json carries an HMAC-SHA256
signature made with a shared secret (ARTIFACT_SIGNING_KEY). Models are pickles: loading one is code execution,
so remote artifacts are refused unless the signature verifies. A tampered bucket cannot push code into Sfumato
without the signing key.
"""
from __future__ import annotations

import hashlib
import hmac
import json
import os
import re
import shutil
import tempfile
from datetime import datetime, timezone
from pathlib import Path

SAFE_NAME = re.compile(r"^[A-Za-z0-9][A-Za-z0-9._-]{0,127}$")


class ArtifactError(Exception):
    pass


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


# ------------------------------------------------------------------------------------------------ stores
class LocalStore:
    kind = "local"

    def __init__(self, root: str | Path):
        self.root = Path(root)

    def describe(self) -> str:
        return f"file://{self.root}"

    def read_text(self, key: str) -> str | None:
        p = self.root / key
        return p.read_text(encoding="utf-8") if p.is_file() else None

    def write_text(self, key: str, text: str) -> None:
        p = self.root / key
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(text, encoding="utf-8")

    def download_file(self, key: str, dest: Path) -> None:
        shutil.copyfile(self.root / key, dest)

    def upload_file(self, src: Path, key: str) -> None:
        p = self.root / key
        p.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(src, p)


class GCSStore:
    kind = "gcs"

    def __init__(self, bucket: str, prefix: str = "", client=None):
        if client is None:
            try:
                from google.cloud import storage
            except ImportError as e:  # pragma: no cover
                raise ArtifactError("pip install google-cloud-storage to use gs:// stores") from e
            client = storage.Client()  # credentials: GOOGLE_APPLICATION_CREDENTIALS or the environment
        self.bucket_name, self.prefix = bucket, prefix.strip("/")
        self.bucket = client.bucket(bucket)

    def describe(self) -> str:
        return f"gs://{self.bucket_name}/{self.prefix}".rstrip("/")

    def _key(self, key: str) -> str:
        return f"{self.prefix}/{key}" if self.prefix else key

    def read_text(self, key: str) -> str | None:
        blob = self.bucket.blob(self._key(key))
        return blob.download_as_text() if blob.exists() else None

    def write_text(self, key: str, text: str) -> None:
        self.bucket.blob(self._key(key)).upload_from_string(text, content_type="application/json")

    def download_file(self, key: str, dest: Path) -> None:
        self.bucket.blob(self._key(key)).download_to_filename(str(dest))

    def upload_file(self, src: Path, key: str) -> None:
        self.bucket.blob(self._key(key)).upload_from_filename(str(src))


class HFStore:
    kind = "huggingface"

    def __init__(self, repo: str, prefix: str = "", token: str | None = None):
        self.repo, self.prefix, self.token = repo, prefix.strip("/"), token

    def describe(self) -> str:
        return f"hf://{self.repo}/{self.prefix}".rstrip("/")

    def _key(self, key: str) -> str:
        return f"{self.prefix}/{key}" if self.prefix else key

    def read_text(self, key: str) -> str | None:
        from huggingface_hub import hf_hub_download
        from huggingface_hub.errors import EntryNotFoundError

        try:
            p = hf_hub_download(self.repo, self._key(key), repo_type="dataset", token=self.token, force_download=True)
        except EntryNotFoundError:
            return None
        return Path(p).read_text(encoding="utf-8")

    def write_text(self, key: str, text: str) -> None:
        from huggingface_hub import HfApi

        HfApi(token=self.token).upload_file(path_or_fileobj=text.encode(), path_in_repo=self._key(key),
                                            repo_id=self.repo, repo_type="dataset")

    def download_file(self, key: str, dest: Path) -> None:
        from huggingface_hub import hf_hub_download

        shutil.copyfile(hf_hub_download(self.repo, self._key(key), repo_type="dataset", token=self.token,
                                        force_download=True), dest)

    def upload_file(self, src: Path, key: str) -> None:
        from huggingface_hub import HfApi

        api = HfApi(token=self.token)
        api.create_repo(self.repo, repo_type="dataset", private=True, exist_ok=True)
        api.upload_file(path_or_fileobj=str(src), path_in_repo=self._key(key), repo_id=self.repo, repo_type="dataset")


def open_store(uri: str, hf_token: str | None = None):
    """gs://bucket/prefix | hf://user/repo[/prefix] | file:///folder | /folder"""
    if uri.startswith("gs://"):
        bucket, _, prefix = uri[5:].partition("/")
        if not bucket:
            raise ArtifactError(f"Invalid gs:// URI: {uri}")
        return GCSStore(bucket, prefix)
    if uri.startswith("hf://"):
        parts = uri[5:].split("/")
        if len(parts) < 2:
            raise ArtifactError(f"Invalid hf:// URI (expected hf://user/repo[/prefix]): {uri}")
        return HFStore("/".join(parts[:2]), "/".join(parts[2:]), hf_token)
    return LocalStore(uri[7:] if uri.startswith("file://") else uri)


# --------------------------------------------------------------------------------------- sign / publish / fetch
def _canonical(meta: dict) -> bytes:
    return json.dumps(meta, sort_keys=True, separators=(",", ":")).encode()


def sign(meta: dict, key: str) -> str:
    return hmac.new(key.encode(), _canonical(meta), hashlib.sha256).hexdigest()


def publish(store, name: str, folder: str | Path, signing_key: str | None = None, version: str | None = None,
            extra: dict | None = None) -> str:
    """Upload every file of `folder` as a new immutable version, then move the LATEST pointer."""
    if not SAFE_NAME.match(name):
        raise ArtifactError(f"Invalid artifact name: {name!r}")
    folder = Path(folder)
    version = version or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    files = {f.name: sha256_file(f) for f in sorted(folder.iterdir()) if f.is_file() and f.name != "ARTIFACT.json"}
    if not files:
        raise ArtifactError(f"Nothing to publish in {folder}")
    bad = [n for n in files if not SAFE_NAME.match(n)]
    if bad:
        raise ArtifactError(f"Unsupported file names: {bad}")
    meta = {"name": name, "version": version, "created_at": datetime.now(timezone.utc).isoformat(),
            "files": files, **(extra or {})}
    for fname in files:
        store.upload_file(folder / fname, f"{name}/{version}/{fname}")
    doc = {"meta": meta, "signature": sign(meta, signing_key) if signing_key else None}
    store.write_text(f"{name}/{version}/ARTIFACT.json", json.dumps(doc, indent=1))
    store.write_text(f"{name}/LATEST.json", json.dumps({"version": version}))  # last: readers never see a partial version
    return version


def fetch_latest(store, name: str, dest_root: str | Path, signing_key: str | None = None,
                 require_signature: bool = True) -> tuple[str, Path, dict] | None:
    """Download (and verify) the newest version. Returns (version, folder, meta) or None if nothing is published."""
    latest = store.read_text(f"{name}/LATEST.json")
    if latest is None:
        return None
    version = str(json.loads(latest).get("version", ""))
    if not SAFE_NAME.match(version):
        raise ArtifactError(f"Invalid version in LATEST.json: {version!r}")
    raw = store.read_text(f"{name}/{version}/ARTIFACT.json")
    if raw is None:
        raise ArtifactError(f"{name}/{version}/ARTIFACT.json is missing in {store.describe()}")
    doc = json.loads(raw)
    meta, signature = doc.get("meta") or {}, doc.get("signature")
    if meta.get("name") != name or meta.get("version") != version:
        raise ArtifactError("ARTIFACT.json does not match the requested name/version")
    if require_signature and not signing_key:
        raise ArtifactError("ARTIFACT_SIGNING_KEY is required to load remote artifacts (they contain pickles)")
    if signing_key and signature is not None:  # always verified when a key is known, even for trusted local folders
        if not hmac.compare_digest(sign(meta, signing_key), signature):
            raise ArtifactError(f"Invalid signature for {name}/{version}: artifact rejected")
    elif require_signature:
        raise ArtifactError(f"{name}/{version} is not signed: artifact rejected")
    target = Path(dest_root) / name / version
    if (target / ".verified").exists():
        return version, target, meta
    tmp = Path(tempfile.mkdtemp(prefix=f".{name}-", dir=_ensure(Path(dest_root) / name)))
    try:
        for fname, digest in (meta.get("files") or {}).items():
            if not SAFE_NAME.match(fname):
                raise ArtifactError(f"Unsafe file name in ARTIFACT.json: {fname!r}")
            store.download_file(f"{name}/{version}/{fname}", tmp / fname)
            if sha256_file(tmp / fname) != digest:
                raise ArtifactError(f"Checksum mismatch for {fname}: artifact rejected")
        (tmp / ".verified").write_text(version, encoding="utf-8")
        if target.exists():
            shutil.rmtree(target)
        os.replace(tmp, target)
    except Exception:
        shutil.rmtree(tmp, ignore_errors=True)
        raise
    return version, target, meta


def _ensure(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p

`guide_features.py` — generated from `backend/app/ml/guide_features.py` in the Sfumato repository.

In [ ]:
%%writefile guide_features.py
"""Incident-level features in the vocabulary of the Microsoft GUIDE dataset.

GUIDE (Microsoft Security Incident Prediction, Kaggle, CDLA-Permissive-2.0) holds ~1M real SOC incidents graded
TruePositive / BenignPositive / FalsePositive by customer analysts. Most of its columns are anonymised
identifiers (DetectorId, AlertTitle, DeviceId, …) that no other system can reproduce, so the model only uses what
is *portable*: the alert category, the MITRE ATT&CK techniques, the types of entities involved and the time of day.
Everything is matched through a vocabulary learnt from the data and normalised names, so "initial-access" in a
Sfumato alert matches "InitialAccess" in GUIDE.

Standalone (standard library + numpy): this exact file is used for training (Kaggle notebook) and inference
(Sfumato backend), so training and serving cannot drift apart.
"""
from __future__ import annotations

import math
import re
from datetime import datetime
from typing import Iterable

import numpy as np

LABELS = ["true_positive", "false_positive", "benign_positive"]
GRADE_MAP = {"truepositive": "true_positive", "falsepositive": "false_positive", "benignpositive": "benign_positive"}
MAX_TECHNIQUES = 8


def norm_key(value) -> str:
    return re.sub(r"[^a-z0-9]", "", str(value).lower())


def split_techniques(value) -> list[str]:
    """'T1059.001;T1566' -> ['T1059.001', 'T1059', 'T1566'] (sub-techniques also count as their parent)."""
    out: list[str] = []
    for t in re.findall(r"T\d{4}(?:\.\d{3})?", str(value or "").upper()):
        out.append(t)
        base = t.split(".")[0]
        if base != t:
            out.append(base)
    return out


def build_vocab(categories: dict[str, int], techniques: dict[str, int], entities: dict[str, int],
                max_categories: int = 60, max_techniques: int = 150, max_entities: int = 40) -> dict:
    def top(counter: dict[str, int], n: int, keyfn=lambda x: x) -> list[str]:
        seen, out = set(), []
        for k, _ in sorted(counter.items(), key=lambda kv: (-kv[1], kv[0])):
            nk = keyfn(k)
            if nk and nk not in seen:
                seen.add(nk)
                out.append(nk)
            if len(out) >= n:
                break
        return out

    return {"categories": top(categories, max_categories, norm_key),
            "techniques": top(techniques, max_techniques, lambda t: t.upper()),
            "entities": top(entities, max_entities, norm_key)}


def feature_names(vocab: dict) -> list[str]:
    return ([f"cat:{c}" for c in vocab["categories"]] + [f"tech:{t}" for t in vocab["techniques"]]
            + [f"ent:{e}" for e in vocab["entities"]]
            + ["n_categories", "n_techniques", "n_entity_types", "no_category", "no_technique",
               "hour_sin", "hour_cos", "weekend"])


def _parse_ts(ts) -> datetime | None:
    if ts is None or ts == "":
        return None
    if isinstance(ts, datetime):
        return ts
    try:
        return datetime.fromisoformat(str(ts).replace("Z", "+00:00"))
    except ValueError:
        return None


def featurize(vocab: dict, categories: Iterable[str], techniques: Iterable[str], entities: Iterable[str],
              timestamp=None) -> tuple[np.ndarray, dict]:
    """Returns (feature vector, coverage). coverage tells how many of the provided signals the vocabulary knows."""
    cats = {norm_key(c) for c in categories if norm_key(c)}
    techs = {t for raw in techniques for t in split_techniques(raw)}
    ents = {norm_key(e) for e in entities if norm_key(e)}
    ci = {c: i for i, c in enumerate(vocab["categories"])}
    ti = {t: i for i, t in enumerate(vocab["techniques"])}
    ei = {e: i for i, e in enumerate(vocab["entities"])}
    nc, nt, ne = len(ci), len(ti), len(ei)
    x = np.zeros(nc + nt + ne + 8, dtype=np.float32)
    for c in cats:
        if c in ci:
            x[ci[c]] = 1
    for t in techs:
        if t in ti:
            x[nc + ti[t]] = 1
    for e in ents:
        if e in ei:
            x[nc + nt + ei[e]] = 1
    base_techs = {t for t in techs if "." not in t}
    o = nc + nt + ne
    x[o] = len(cats)
    x[o + 1] = min(len(base_techs), MAX_TECHNIQUES)
    x[o + 2] = len(ents)
    x[o + 3] = 0 if cats else 1
    x[o + 4] = 0 if techs else 1
    dt = _parse_ts(timestamp)
    if dt is not None:
        x[o + 5] = math.sin(2 * math.pi * dt.hour / 24)
        x[o + 6] = math.cos(2 * math.pi * dt.hour / 24)
        x[o + 7] = 1 if dt.weekday() >= 5 else 0
    matched = {"categories": [c for c in cats if c in ci], "techniques": [t for t in techs if t in ti],
               "entities": [e for e in ents if e in ei]}
    provided = {"categories": len(cats), "techniques": len(techs), "entities": len(ents)}
    present = [k for k, n in provided.items() if n]
    coverage = (sum(1 for k in present if matched[k]) / len(present)) if present else 0.0
    return x, {"coverage": round(coverage, 3), "provided": provided, "matched": {k: sorted(v) for k, v in matched.items()}}

`train_guide.py` — generated from `ml/guide/train_guide.py` in the Sfumato repository.

In [ ]:
%%writefile train_guide.py
#!/usr/bin/env python3
"""Train the GUIDE triage model (TruePositive / FalsePositive / BenignPositive) and publish it to a bucket.

Runs on a free Kaggle notebook (see sfumato_guide_kaggle.ipynb) or anywhere with the CSV files:

    python train_guide.py --csv GUIDE_Train.csv GUIDE_Test.csv --out guide_out \
        --publish-to gs://my-bucket/sfumato      # needs ARTIFACT_SIGNING_KEY + GCS credentials

Method
------
* Rows (evidence items) are aggregated per incident: categories, MITRE techniques, entity types, first timestamp.
  Only portable signals are used (see backend/app/ml/guide_features.py); anonymised IDs are ignored.
* Leakage control: an incident is one atomic example; whole *organisations* (15 %) are held out as the test set, so
  the reported metrics measure generalisation to organisations never seen in training — the real situation of a
  new SOC using Sfumato. Validation (for model selection) is a stratified split of the remaining incidents.
* Candidates: Logistic Regression (baseline), Random Forest, XGBoost; selection by validation macro-F1, the simpler
  model wins ties within 0.005. The report adds the binary view "threat (TP) vs non-threat (FP + BP)".
* An IsolationForest fitted on the training data lets the service abstain on out-of-distribution inputs.
* Dataset: Microsoft Security Incident Prediction (GUIDE), CDLA-Permissive-2.0, https://www.kaggle.com/datasets/Microsoft/microsoft-security-incident-prediction
"""
from __future__ import annotations

import argparse
import json
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

HERE = Path(__file__).resolve().parent
for cand in (HERE, HERE.parent.parent / "backend" / "app" / "ml", HERE.parent.parent / "backend" / "app" / "storage"):
    if cand.exists() and str(cand) not in sys.path:
        sys.path.insert(0, str(cand))
import artifacts  # noqa: E402
import guide_features as gf  # noqa: E402

DATASET = {"name": "Microsoft Security Incident Prediction (GUIDE)", "license": "CDLA-Permissive-2.0",
           "url": "https://www.kaggle.com/datasets/Microsoft/microsoft-security-incident-prediction"}
COMPLEXITY = {"logistic_regression": 0, "random_forest": 1, "xgboost": 2}
USECOLS = {"IncidentId", "OrgId", "Timestamp", "Category", "MitreTechniques", "EntityType", "IncidentGrade"}
REQUIRED = {"IncidentId", "IncidentGrade"}


def aggregate(csv_paths: list[str], max_rows: int | None, log) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    metas, pairs = [], {"cat": [], "tech": [], "ent": []}
    rows = 0
    for path in csv_paths:
        for chunk in pd.read_csv(path, usecols=lambda c: c in USECOLS, chunksize=1_000_000, dtype=str):
            missing = REQUIRED - set(chunk.columns)
            if missing:
                raise SystemExit(f"{path}: missing column(s) {sorted(missing)} — is this a GUIDE file?")
            if max_rows is not None and rows + len(chunk) > max_rows:
                chunk = chunk.iloc[: max_rows - rows]
            rows += len(chunk)
            if "OrgId" not in chunk:
                chunk["OrgId"] = "unknown"
            if "Timestamp" not in chunk:
                chunk["Timestamp"] = None
            metas.append(chunk.groupby("IncidentId").agg(org=("OrgId", "first"), grade=("IncidentGrade", "first"),
                                                         ts=("Timestamp", "min")))
            if "Category" in chunk:
                pairs["cat"].append(chunk[["IncidentId", "Category"]].dropna().drop_duplicates())
            if "MitreTechniques" in chunk:
                t = chunk[["IncidentId", "MitreTechniques"]].dropna().drop_duplicates()
                t["Tech"] = t["MitreTechniques"].map(gf.split_techniques)
                pairs["tech"].append(t[["IncidentId", "Tech"]].explode("Tech").dropna().drop_duplicates())
            if "EntityType" in chunk:
                pairs["ent"].append(chunk[["IncidentId", "EntityType"]].dropna().drop_duplicates())
            log(f"  {rows:,} rows read")
            if max_rows is not None and rows >= max_rows:
                break
    meta = pd.concat(metas).groupby(level=0).agg(org=("org", "first"), grade=("grade", "first"), ts=("ts", "min"))
    meta["label"] = meta["grade"].map(lambda g: gf.GRADE_MAP.get(gf.norm_key(g)) if isinstance(g, str) else None)
    meta = meta.dropna(subset=["label"])
    merged = {k: (pd.concat(v).drop_duplicates() if v else pd.DataFrame(columns=["IncidentId", "x"]))
              for k, v in pairs.items()}
    return meta, merged


def build_matrix(meta: pd.DataFrame, pairs: dict[str, pd.DataFrame], log):
    cat, tech, ent = pairs["cat"], pairs["tech"], pairs["ent"]
    in_meta = lambda df: df[df["IncidentId"].isin(meta.index)]  # noqa: E731
    cat, tech, ent = in_meta(cat), in_meta(tech), in_meta(ent)
    cat_col = cat.columns[1] if len(cat) else None
    vocab = gf.build_vocab(cat[cat_col].value_counts().to_dict() if cat_col else {},
                           tech["Tech"].value_counts().to_dict() if len(tech) else {},
                           ent[ent.columns[1]].value_counts().to_dict() if len(ent) else {})
    by = {k: (df.groupby("IncidentId")[df.columns[1]].agg(list) if len(df) else pd.Series(dtype=object))
          for k, df in (("cat", cat), ("tech", tech), ("ent", ent))}
    X = np.zeros((len(meta), len(gf.feature_names(vocab))), dtype=np.float32)
    empty: list = []
    for i, (iid, ts) in enumerate(zip(meta.index, meta["ts"])):
        X[i], _ = gf.featurize(vocab, by["cat"].get(iid, empty), by["tech"].get(iid, empty), by["ent"].get(iid, empty), ts)
        if i % 200_000 == 0 and i:
            log(f"  featurised {i:,}/{len(meta):,}")
    return X, vocab


def candidates(fast: bool, n_classes: int):
    from xgboost import XGBClassifier

    return {
        "logistic_regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced")),
        "random_forest": RandomForestClassifier(n_estimators=60 if fast else 200, max_depth=16, min_samples_leaf=3,
                                                class_weight="balanced", n_jobs=-1, random_state=7,
                                                max_samples=0.6),
        "xgboost": XGBClassifier(n_estimators=80 if fast else 300, max_depth=7, learning_rate=0.1, subsample=0.8,
                                 colsample_bytree=0.8, objective="multi:softprob", num_class=n_classes,
                                 eval_metric="mlogloss", tree_method="hist", random_state=7, n_jobs=4),
    }


def metrics(model, X, y) -> dict:
    proba = model.predict_proba(X)
    pred = proba.argmax(axis=1)
    p, r, f, s = precision_recall_fscore_support(y, pred, labels=range(len(gf.LABELS)), zero_division=0)
    present = len(np.unique(y)) == len(gf.LABELS)
    threat = (y == 0)
    pred_threat = pred == 0
    tp = int((threat & pred_threat).sum())
    return {
        "macro_f1": round(float(f1_score(y, pred, average="macro")), 4),
        "accuracy": round(float((pred == y).mean()), 4),
        "roc_auc_ovr_macro": round(float(roc_auc_score(y, proba, multi_class="ovr", average="macro")), 4) if present else None,
        "per_class": {gf.LABELS[i]: {"precision": round(float(p[i]), 4), "recall": round(float(r[i]), 4),
                                     "f1": round(float(f[i]), 4), "support": int(s[i])} for i in range(len(gf.LABELS))},
        "confusion_matrix": {"labels": gf.LABELS, "matrix": confusion_matrix(y, pred, labels=range(len(gf.LABELS))).tolist()},
        "binary_threat_vs_non_threat": {
            "roc_auc": round(float(roc_auc_score(threat, proba[:, 0])), 4) if 0 < threat.sum() < len(y) else None,
            "threat_precision": round(tp / max(1, int(pred_threat.sum())), 4),
            "threat_recall": round(tp / max(1, int(threat.sum())), 4),
            "non_threat_precision": round(int((~threat & ~pred_threat).sum()) / max(1, int((~pred_threat).sum())), 4),
            "non_threat_recall": round(int((~threat & ~pred_threat).sum()) / max(1, int((~threat).sum())), 4)},
    }


def importance(name: str, model, names: list[str]) -> list[dict]:
    vals = np.abs(model[-1].coef_).mean(axis=0) if name == "logistic_regression" else model.feature_importances_
    vals = vals / (vals.sum() or 1)
    return [{"feature": names[i], "importance": round(float(vals[i]), 4)} for i in np.argsort(vals)[::-1][:40]]


def train(csv_paths: list[str], out_dir: str, max_rows: int | None = None, seed: int = 42, fast: bool = False,
          log=print) -> dict:
    t0 = time.time()
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    log(f"Aggregating {len(csv_paths)} file(s) per incident…")
    meta, pairs = aggregate(csv_paths, max_rows, log)
    if len(meta) < 200:
        raise SystemExit(f"Only {len(meta)} graded incidents found: need more data")
    X, vocab = build_matrix(meta, pairs, log)
    y = meta["label"].map({l: i for i, l in enumerate(gf.LABELS)}).to_numpy()
    names = gf.feature_names(vocab)

    rng = np.random.default_rng(seed)
    orgs = meta["org"].astype(str).unique()
    split_note = "organisation hold-out (15 % of organisations)"
    if len(orgs) >= 8:
        test_orgs = set(rng.choice(orgs, size=max(2, int(0.15 * len(orgs))), replace=False))
        is_test = meta["org"].astype(str).isin(test_orgs).to_numpy()
        tr_idx, te_idx = np.where(~is_test)[0], np.where(is_test)[0]
    else:  # too few organisations to hold any out
        tr_idx, te_idx = train_test_split(np.arange(len(y)), test_size=0.15, stratify=y, random_state=seed)
        split_note = "stratified incident split (fewer than 8 organisations in the data)"
    tr_idx, va_idx = train_test_split(tr_idx, test_size=0.15, stratify=y[tr_idx], random_state=seed)
    Xtr, ytr, Xva, yva, Xte, yte = X[tr_idx], y[tr_idx], X[va_idx], y[va_idx], X[te_idx], y[te_idx]
    log(f"train {len(ytr):,} / validation {len(yva):,} / test {len(yte):,} incidents ({split_note})")

    models = candidates(fast, len(gf.LABELS))
    evaluated = {}
    w = compute_sample_weight("balanced", ytr)
    for name, model in models.items():
        ts = time.time()
        if name == "xgboost":
            model.fit(Xtr, ytr, sample_weight=w)
        else:
            model.fit(Xtr, ytr)
        evaluated[name] = {"validation": metrics(model, Xva, yva), "fit_seconds": round(time.time() - ts, 1)}
        log(f"  {name}: validation macro-F1 {evaluated[name]['validation']['macro_f1']}")
    best = max(evaluated, key=lambda n: (evaluated[n]["validation"]["macro_f1"], -COMPLEXITY[n]))
    simpler = [n for n in evaluated if COMPLEXITY[n] < COMPLEXITY[best]
               and evaluated[best]["validation"]["macro_f1"] - evaluated[n]["validation"]["macro_f1"] <= 0.005]
    if simpler:
        best = min(simpler, key=lambda n: COMPLEXITY[n])
    model = models[best]
    test = metrics(model, Xte, yte)

    sub = Xtr[rng.choice(len(Xtr), size=min(len(Xtr), 200_000), replace=False)]
    iso = IsolationForest(n_estimators=100, random_state=seed, n_jobs=-1).fit(sub)
    ood_threshold = float(np.percentile(iso.score_samples(sub), 0.5))
    version = f"{datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S')}-{best}"
    artifact = {"model": model, "model_name": best, "version": version, "vocab": vocab, "feature_names": names,
                "labels": gf.LABELS, "isolation_forest": iso, "ood_threshold": ood_threshold,
                "sklearn_version": sklearn.__version__, "trained_at": datetime.now(timezone.utc).isoformat(),
                "dataset": DATASET}
    joblib.dump(artifact, out / "guide_model.joblib", compress=3)
    report = {
        "model_name": best, "version": version, "trained_at": artifact["trained_at"], "dataset": DATASET,
        "data": {"rows_read_limit": max_rows, "incidents": int(len(y)), "organisations": int(len(orgs)),
                 "class_balance": {gf.LABELS[i]: int((y == i).sum()) for i in range(len(gf.LABELS))},
                 "split": {"method": split_note, "train": int(len(ytr)), "validation": int(len(yva)), "test": int(len(yte))},
                 "features": len(names), "vocabulary": {k: len(v) for k, v in vocab.items()}},
        "selection": {"criterion": "validation macro-F1, simpler model wins ties within 0.005", "candidates": evaluated},
        "test": test, "feature_importance": importance(best, model, names),
        "abstention": {"min_coverage": 0.34, "min_probability": 0.5, "ood_threshold": round(ood_threshold, 4),
                       "ood": "IsolationForest score below the 0.5th percentile of training scores"},
        "limitations": [
            "Only portable signals (category, ATT&CK techniques, entity types, hour) are used: anonymised GUIDE "
            "identifiers cannot be reproduced from a Sfumato alert, so this is a coarse, second-opinion model.",
            "GUIDE comes from Microsoft customers' detections (Defender / Sentinel); alerts from other sources can "
            "be out of distribution — the service abstains when its coverage or novelty checks fail.",
            "Labels are analyst decisions of those customers, not ground truth."],
        "training_seconds": round(time.time() - t0, 1), "sklearn_version": sklearn.__version__}
    (out / "metrics.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    log(f"Done in {report['training_seconds']} s — best: {best}, test macro-F1 {test['macro_f1']}, "
        f"threat-vs-rest ROC-AUC {test['binary_threat_vs_non_threat']['roc_auc']}")
    return report


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--csv", nargs="+", required=True, help="GUIDE_Train.csv [GUIDE_Test.csv]")
    ap.add_argument("--out", default="guide_out")
    ap.add_argument("--max-rows", type=int, default=None, help="limit rows read (quick runs)")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--fast", action="store_true", help="smaller models (smoke tests)")
    ap.add_argument("--publish-to", help="gs://bucket/prefix | hf://user/repo | folder (needs ARTIFACT_SIGNING_KEY)")
    args = ap.parse_args(argv)
    report = train(args.csv, args.out, args.max_rows, args.seed, args.fast)
    if args.publish_to:
        key = os.environ.get("ARTIFACT_SIGNING_KEY")
        if not key:
            raise SystemExit("Set ARTIFACT_SIGNING_KEY (same value in Sfumato's .env): remote models must be signed")
        store = artifacts.open_store(args.publish_to, os.environ.get("HF_TOKEN"))
        version = artifacts.publish(store, "guide_triage", args.out, signing_key=key,
                                    extra={"model": report["model_name"], "macro_f1": report["test"]["macro_f1"],
                                           "dataset": DATASET["name"]})
        print(f"Published guide_triage/{version} to {store.describe()}")
        print(f"In Sfumato's .env:  ML_ARTIFACT_URI={args.publish_to}")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
import glob
csvs = sorted(glob.glob("/kaggle/input/**/GUIDE_*.csv", recursive=True))
print(csvs)
rows = "" if MAX_ROWS is None else f"--max-rows {MAX_ROWS}"
!python train_guide.py --csv {" ".join(csvs)} --out /kaggle/working/guide_out {rows} --publish-to {BUCKET_URI}

In [ ]:
import json
r = json.load(open("/kaggle/working/guide_out/metrics.json"))
print(r["model_name"], r["data"]["split"])
print("test macro-F1:", r["test"]["macro_f1"], "| threat vs non-threat:", r["test"]["binary_threat_vs_non_threat"])
print(r["test"]["confusion_matrix"])

**Then, in Sfumato's `.env`:**
```
ML_ARTIFACT_URI=gs://YOUR-BUCKET/sfumato
ARTIFACT_SIGNING_KEY=<same value as the Kaggle secret>
GOOGLE_APPLICATION_CREDENTIALS=/run/secrets/gcs-reader.json   # service account with READ-ONLY access (roles/storage.objectViewer)
```
Restart Sfumato or call `POST /api/v1/ml/guide/sync`.